# 01 — Data Ingestion and Chunking

Thin demo layer over `scripts/ingest.py`.  All configuration lives in `scripts/config.py`.

Shows:
- how the ingestion module is configured
- how PDF pages are extracted
- how chunks are created
- how `chunks.jsonl` is generated

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\aminl\Desktop\PORTFOLIO Projects\final list of projects\aegis-rag - fraud


In [2]:
# All constants now come from config — no duplication across scripts.
from scripts.config import PDF_DIR, TEXT_DIR, CHUNKS_FILE, CHUNKING_METHOD
from scripts.ingest import extract_pdf_pages, build_chunks_from_pages, main as ingest_main

print("PDF_DIR       :", PDF_DIR)
print("TEXT_DIR      :", TEXT_DIR)
print("CHUNKS_FILE   :", CHUNKS_FILE)
print("CHUNKING_METHOD:", CHUNKING_METHOD)

PDF_DIR       : C:\Users\aminl\Desktop\PORTFOLIO Projects\final list of projects\aegis-rag - fraud\data\pdfs
TEXT_DIR      : C:\Users\aminl\Desktop\PORTFOLIO Projects\final list of projects\aegis-rag - fraud\data\texts
CHUNKS_FILE   : C:\Users\aminl\Desktop\PORTFOLIO Projects\final list of projects\aegis-rag - fraud\data\texts\chunks.jsonl
CHUNKING_METHOD: semantic


## Preview available PDFs

In [3]:
pdf_files = sorted(PDF_DIR.glob("*.pdf"))
len(pdf_files), [p.name for p in pdf_files]

(6,
 ['01_FINTRAC_LCTR_Guidance.pdf',
  '02_FINTRAC_24hour_Rule.pdf',
  '03_FINTRAC_EFT_Reporting.pdf',
  '04_OSFI_E21_Operational_Risk.pdf',
  '05_OSFI_B13_Cyber_Risk.pdf',
  '06_OSFI_Third_Party_Risk.pdf'])

## Preview raw extracted pages from one PDF

In [4]:
sample_pdf = pdf_files[0]
pages = extract_pdf_pages(sample_pdf)
len(pages), pages[0]

(102,
 {'source': '01_FINTRAC_LCTR_Guidance.pdf',
  'page': 1,
  'text': "Canada.ca\ue080 FINTRAC \ue080 Obligations and guidance\nFINTRAC's compliance guidance\nReporting large cash transactions to\nFINTRAC\nFrom: Financial Transactions and Reports Analysis Centre of Canada\n(FINTRAC)\nThis guidance explains the requirement to report large cash transactions to\nFINTRAC.\nNote:\nThroughout this guidance, references to dollar amounts (such as\n$10,000) are in Canadian dollars unless otherwise specified.\nThe examples and scenarios are meant to help explain reporting\nrequirements.\nThe details used in these examples and scenarios such as names of\npersons, names of entities, addresses, phone numbers and email\naddresses are fictitious.\nIn this guidance\n1. Who must comply\n2. What is a large cash transaction\n3. When to submit a Large Cash Transaction Report\n4. How to submit a report to FINTRAC\n5. The form for reporting large cash transactions\n6. Other requirements associated with l

## Preview chunks from one PDF

In [5]:
sample_chunks = build_chunks_from_pages(pages)
len(sample_chunks), sample_chunks[:2]

(167,
 [{'chunk_id': '01_FINTRAC_LCTR_Guidance_p1_c1',
   'source': '01_FINTRAC_LCTR_Guidance.pdf',
   'page': 1,
   'title': '01 Fintrac Lctr Guidance',
   'section': None,
   'text': "Canada.ca\ue080 FINTRAC \ue080 Obligations and guidance\nFINTRAC's compliance guidance\nReporting large cash transactions to\nFINTRAC\nFrom: Financial Transactions and Reports Analysis Centre of Canada\n(FINTRAC)\nThis guidance explains the requirement to report large cash transactions to\nFINTRAC.\nNote:\nThroughout this guidance, references to dollar amounts (such as\n$10,000) are in Canadian dollars unless otherwise specified.\nThe examples and scenarios are meant to help explain reporting\nrequirements.\nThe details used in these examples and scenarios such as names of\npersons, names of entities, addresses, phone numbers and email\naddresses are fictitious.\nIn this guidance\n1. Who must comply\n2. What is a large cash transaction\n3. When to submit a Large Cash Transaction Report\n4. How to submit

## Run the full ingestion pipeline

In [6]:
ingest_main()

Processed PDFs : 6
Extracted pages: 415
Saved chunks   : 800
Output file    : C:\Users\aminl\Desktop\PORTFOLIO Projects\final list of projects\aegis-rag - fraud\data\texts\chunks.jsonl
Chunking method: semantic


## Inspect the saved JSONL output

In [7]:
import json

rows = []
with CHUNKS_FILE.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        rows.append(json.loads(line))
rows

[{'chunk_id': '01_FINTRAC_LCTR_Guidance_p1_c1',
  'source': '01_FINTRAC_LCTR_Guidance.pdf',
  'page': 1,
  'title': '01 Fintrac Lctr Guidance',
  'section': None,
  'text': "Canada.ca\ue080 FINTRAC \ue080 Obligations and guidance\nFINTRAC's compliance guidance\nReporting large cash transactions to\nFINTRAC\nFrom: Financial Transactions and Reports Analysis Centre of Canada\n(FINTRAC)\nThis guidance explains the requirement to report large cash transactions to\nFINTRAC.\nNote:\nThroughout this guidance, references to dollar amounts (such as\n$10,000) are in Canadian dollars unless otherwise specified.\nThe examples and scenarios are meant to help explain reporting\nrequirements.\nThe details used in these examples and scenarios such as names of\npersons, names of entities, addresses, phone numbers and email\naddresses are fictitious.\nIn this guidance\n1. Who must comply\n2. What is a large cash transaction\n3. When to submit a Large Cash Transaction Report\n4. How to submit a report to

In [8]:
#  BUILD BM25 INDEX

from scripts.build_index import main as build_index_main

build_index_main()

Loaded 800 chunks from C:\Users\aminl\Desktop\PORTFOLIO Projects\final list of projects\aegis-rag - fraud\data\texts\chunks.jsonl
Building BM25 index...
Saved BM25 index → C:\Users\aminl\Desktop\PORTFOLIO Projects\final list of projects\aegis-rag - fraud\data\embeddings\bm25_index.pkl
Building Chroma dense index...
Saved Chroma collection → C:\Users\aminl\Desktop\PORTFOLIO Projects\final list of projects\aegis-rag - fraud\data\embeddings\chroma
Saved chunk metadata → C:\Users\aminl\Desktop\PORTFOLIO Projects\final list of projects\aegis-rag - fraud\data\embeddings\chunks_metadata.json

Done. Indexed 800 chunks.
Embedding model    : intfloat/e5-large-v2
Chroma collection  : hotel_chunks
